# Cross-Validation and Model Selection — Intermediate ML Lab
Amazon SageMaker AI Notebooks | Binary classification

**Dataset:** Wisconsin Diagnostic Breast Cancer benchmark (569 rows, 30 numeric features). Educational use only; not a clinical decision tool.

Learning outcomes: stratified K-fold CV, model comparison, leakage-safe pipelines, hyperparameter search, held-out test evaluation, ROC-AUC/PR-AUC, permutation importance and model persistence.

Upload this notebook and `wdbc_dataset.csv` into the same SageMaker JupyterLab folder.

## 1. Install dependencies

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib seaborn joblib scipy

## 2. Imports

In [ ]:
import os, joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
 average_precision_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.inspection import permutation_importance
from scipy.stats import randint
RANDOM_STATE=42
pd.set_option("display.max_columns", 40)

## 3. Load and inspect data

In [ ]:
data=pd.read_csv("wdbc_dataset.csv")
display(data.head())
print("Shape:",data.shape)
display(data.target.value_counts().to_frame("count"))
print("Missing values:",data.isna().sum().sum())
X=data.drop(columns="target")
y=data.target.map({"malignant":1,"benign":0}) # positive class = malignant
print("Features:",X.shape[1]," Positive rate:",round(y.mean(),3))

## 4. Hold out the final test set
The test partition is not used for model selection or hyperparameter tuning.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
 X,y,test_size=0.20,stratify=y,random_state=RANDOM_STATE)
print(X_train.shape,X_test.shape)

## 5. Define stratified 5-fold cross-validation
Each training observation is used for validation once. Stratification preserves class proportions.

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)

## 6. Compare model families
Logistic Regression is a scaled linear baseline; Random Forest captures nonlinear interactions; HistGradientBoosting is a boosted-tree model. Preprocessing is inside each Pipeline so it is refitted independently inside every CV fold, preventing leakage.

In [ ]:
models={
 "Logistic Regression":Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler()),
  ("model",LogisticRegression(max_iter=3000,class_weight="balanced",random_state=RANDOM_STATE))]),
 "Random Forest":Pipeline([("imputer",SimpleImputer(strategy="median")),
  ("model",RandomForestClassifier(n_estimators=400,min_samples_leaf=2,class_weight="balanced",
   random_state=RANDOM_STATE,n_jobs=-1))]),
 "Hist Gradient Boosting":Pipeline([("imputer",SimpleImputer(strategy="median")),
  ("model",HistGradientBoostingClassifier(max_iter=150,learning_rate=.08,l2_regularization=1,
   random_state=RANDOM_STATE))])
}
scoring={"accuracy":"accuracy","precision":"precision","recall":"recall","f1":"f1",
 "roc_auc":"roc_auc","average_precision":"average_precision"}
rows=[]
for name,est in models.items():
 s=cross_validate(est,X_train,y_train,cv=cv,scoring=scoring,n_jobs=-1,return_train_score=True)
 r={"Model":name}
 for metric in scoring:
  r[metric+"_mean"]=s["test_"+metric].mean()
  r[metric+"_std"]=s["test_"+metric].std()
 r["train_roc_auc_mean"]=s["train_roc_auc"].mean()
 rows.append(r)
cv_results=pd.DataFrame(rows).sort_values("roc_auc_mean",ascending=False)
display(cv_results.round(3))

In [ ]:
plt.figure(figsize=(9,5))
sns.barplot(data=cv_results,x="roc_auc_mean",y="Model")
plt.xlim(.85,1.0); plt.xlabel("Mean 5-fold validation ROC-AUC")
plt.title("Cross-validated model comparison"); plt.show()

### Interpret the comparison
Compare mean **and standard deviation** across folds. A high standard deviation suggests instability. Compare training and validation scores for possible overfitting. These are model-selection estimates, not final test results.

## 7. Hyperparameter tuning with RandomizedSearchCV
Search 20 parameter combinations using the same stratified CV strategy. Reduce `n_iter` to 8–10 for a faster live run.

In [ ]:
rf=Pipeline([("imputer",SimpleImputer(strategy="median")),
 ("model",RandomForestClassifier(class_weight="balanced",random_state=RANDOM_STATE,n_jobs=-1))])
params={
 "model__n_estimators":randint(250,701),
 "model__max_depth":[None,4,6,8,12,16],
 "model__min_samples_split":randint(2,15),
 "model__min_samples_leaf":randint(1,8),
 "model__max_features":["sqrt","log2",.5,.8]
}
search=RandomizedSearchCV(rf,params,n_iter=20,scoring="roc_auc",cv=cv,n_jobs=-1,
 random_state=RANDOM_STATE,refit=True,return_train_score=True,verbose=1)
search.fit(X_train,y_train)
print("Best CV ROC-AUC:",round(search.best_score_,4))
print("Best parameters:",search.best_params_)
best_model=search.best_estimator_

## 8. Inspect search results and generalization gap

In [ ]:
sr=pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
display(sr[["rank_test_score","mean_train_score","mean_test_score","std_test_score",
 "param_model__n_estimators","param_model__max_depth","param_model__min_samples_leaf",
 "param_model__max_features"]].head(10).round(4))

## 9. Final evaluation on the untouched test set
Do not tune further based on these results.

In [ ]:
y_pred=best_model.predict(X_test)
y_score=best_model.predict_proba(X_test)[:,1]
metrics={"Accuracy":accuracy_score(y_test,y_pred),"Precision (malignant)":precision_score(y_test,y_pred),
 "Recall (malignant)":recall_score(y_test,y_pred),"F1 (malignant)":f1_score(y_test,y_pred),
 "ROC-AUC":roc_auc_score(y_test,y_score),"PR-AUC / Average Precision":average_precision_score(y_test,y_score)}
display(pd.DataFrame(metrics.items(),columns=["Metric","Test score"]).round(4))
print(classification_report(y_test,y_pred,target_names=["benign","malignant"]))

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4))
ConfusionMatrixDisplay.from_predictions(y_test,y_pred,display_labels=["benign","malignant"],cmap="Blues",ax=ax[0],colorbar=False)
RocCurveDisplay.from_predictions(y_test,y_score,ax=ax[1])
PrecisionRecallDisplay.from_predictions(y_test,y_score,ax=ax[2])
plt.tight_layout(); plt.show()

## 10. Threshold analysis
Default classification threshold is 0.50. A lower threshold may increase recall while reducing precision. In a rigorous workflow, select the threshold using validation data, not the final test set. The following is descriptive analysis only.

In [ ]:
from sklearn.metrics import precision_recall_curve
p,r,t=precision_recall_curve(y_test,y_score)
thresholds=pd.DataFrame({"threshold":t,"precision":p[:-1],"recall":r[:-1]})
display(thresholds.iloc[::max(1,len(thresholds)//10)].round(3))

## 11. Model interpretation with permutation importance
Permutation importance estimates how much ROC-AUC drops when one feature is shuffled. Treat this as exploratory interpretation.

In [ ]:
perm=permutation_importance(best_model,X_test,y_test,scoring="roc_auc",n_repeats=15,
 random_state=RANDOM_STATE,n_jobs=-1)
imp=pd.DataFrame({"feature":X_test.columns,"importance_mean":perm.importances_mean,
 "importance_std":perm.importances_std}).sort_values("importance_mean",ascending=False).head(15)
plt.figure(figsize=(9,6)); sns.barplot(data=imp,x="importance_mean",y="feature")
plt.title("Top permutation importances"); plt.xlabel("Mean decrease in ROC-AUC"); plt.show()
display(imp.round(4))

## 12. Save and reload the full pipeline

In [ ]:
MODEL_PATH="wdbc_random_forest_pipeline.joblib"
joblib.dump(best_model,MODEL_PATH)
loaded=joblib.load(MODEL_PATH)
sample=X_test.iloc[[0]]
print("Actual:", "malignant" if y_test.iloc[0]==1 else "benign")
print("Predicted:", "malignant" if loaded.predict(sample)[0]==1 else "benign")
print("Malignant probability:",round(loaded.predict_proba(sample)[0,1],4))
print("Artifact:",MODEL_PATH,round(os.path.getsize(MODEL_PATH)/1024,1),"KB")

## 13. Optional extension: learning curve

In [ ]:
sizes,tr,va=learning_curve(best_model,X_train,y_train,cv=cv,scoring="roc_auc",
 train_sizes=np.linspace(.2,1.0,5),n_jobs=-1)
plt.figure(figsize=(8,5))
plt.plot(sizes,tr.mean(axis=1),marker="o",label="Training ROC-AUC")
plt.plot(sizes,va.mean(axis=1),marker="o",label="CV ROC-AUC")
plt.fill_between(sizes,va.mean(axis=1)-va.std(axis=1),va.mean(axis=1)+va.std(axis=1),alpha=.2)
plt.xlabel("Training examples");plt.ylabel("ROC-AUC");plt.title("Learning curve");plt.legend();plt.show()

## Instructor discussion
1. Why use stratified folds?
2. What leakage occurs if scaling is performed before CV?
3. Why is the test set held out until model selection is complete?
4. What do fold-to-fold standard deviation and train-validation gap indicate?
5. When is PR-AUC more informative than accuracy?
6. How do `n_splits`, `n_iter` and parallel jobs affect runtime?
7. How would this workflow move into SageMaker Processing, Training and Pipelines?

**Limitations:** This is an educational benchmark, not representative deployment data. CV does not remove dataset bias or distribution shift. Nested CV can provide a less biased estimate when tuning is extensive. This notebook uses notebook-local scikit-learn; it does not launch a managed SageMaker training job or endpoint. Do not use the model for medical diagnosis or clinical decisions.